# PowerCo Churn: Cost-Sensitive Classification and Survival Analysis

This notebook builds the full modelling pipeline:

1. Three-way stratified split (train, validation, test).
2. SMOTE-balanced Random Forest inside an `imblearn.Pipeline`.
3. Cost-sensitive threshold selection on the validation fold.
4. 5-fold cross-validation for the cost-optimal threshold.
5. Sensitivity analysis on CLV, campaign cost and retention rate.
6. Model comparison across Logistic Regression, Random Forest and XGBoost.
7. Permutation feature importance on the test fold.
8. Cox proportional-hazards survival model with log-transformed covariates
   and a Schoenfeld residual test of the PH assumption.
9. One-shot evaluation on the test fold.

The shared logic lives under `src/`. The notebook is a thin orchestration
layer.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, recall_score,
    precision_score, roc_auc_score, roc_curve,
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from src.data_loading import load_model_dataset, split_features_target
from src.evaluation import (
    CostMatrix, expected_cost, expected_value,
    optimal_threshold, sweep_thresholds,
)
from src.model import make_smote_rf, three_way_split, cv_threshold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = load_model_dataset()
X, y = split_features_target(df)
print(f"Rows: {len(df):,}    Features: {X.shape[1]}    Churn rate: {y.mean():.2%}")

## 1. Train / Validation / Test Split

A 60 / 20 / 20 stratified split. Threshold selection happens on the
validation fold; the test fold is read only once, at the end.

| Fold | Purpose |
|------|---------|
| Train (60%) | Fit the SMOTE + Random Forest pipeline |
| Validation (20%) | Select the cost-optimal threshold |
| Test (20%) | Final evaluation of the chosen policy |

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X, y, val_size=0.2, test_size=0.2, random_state=RANDOM_STATE,
)
for name, fold in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:>5}: n = {len(fold):>5}    churn = {fold.mean():.2%}")

## 2. SMOTE + Random Forest Pipeline

`make_smote_rf()` wraps SMOTE inside `imblearn.Pipeline` so the resampler
is refit inside every cross-validation split.

In [ ]:
pipe = make_smote_rf(random_state=RANDOM_STATE)
pipe.fit(X_train, y_train)

y_val_prob = pipe.predict_proba(X_val)[:, 1]
y_test_prob = pipe.predict_proba(X_test)[:, 1]

print(f"Validation AUC-ROC: {roc_auc_score(y_val, y_val_prob):.3f}")
print(f"Test AUC-ROC:       {roc_auc_score(y_test, y_test_prob):.3f}")

## 3. Cost-Sensitive Threshold Selection

The cost function is:

```
expected_cost = FN * clv + FP * campaign_cost + TP * (campaign_cost - clv * retention_rate)
# With clv=50_000, campaign_cost=500, retention_rate=0.3 this gives:
#   expected_cost = FN * 50_000 + FP * 500 + TP * (-14_500)
```

The threshold is selected on the validation fold by minimising this cost.

In [ ]:
costs = CostMatrix(clv=50_000, campaign_cost=500, retention_rate=0.3)

t_star, val_cost_star = optimal_threshold(y_val.values, y_val_prob, costs)
val_cost_at_default = expected_cost(y_val.values, y_val_prob, 0.5, costs)

print(f"Optimal threshold (validation): {t_star:.3f}")
print(f"Validation cost at t*       : GBP {val_cost_star:>15,.0f}")
print(f"Validation cost at 0.5      : GBP {val_cost_at_default:>15,.0f}")

In [ ]:
sweep = sweep_thresholds(y_val.values, y_val_prob, costs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(sweep.thresholds, sweep.costs / 1e6, color='C0')
axes[0].axvline(t_star, color='C3', linestyle='--', label=f't* = {t_star:.2f}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Expected cost (GBP M, validation)')
axes[0].set_title('Cost vs. threshold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sweep.thresholds, sweep.recall, label='Recall', color='C2')
axes[1].plot(sweep.thresholds, sweep.precision, label='Precision', color='C1')
axes[1].axvline(t_star, color='C3', linestyle='--')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall and precision vs. threshold')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Important caveat.** On the PowerCo data with the assumed cost parameters
(`CLV = £50k`, `campaign_cost = £500`, `retention_rate = 0.3`) the cost-optimal
threshold lands at the very bottom of the search grid -- around 0.01. The
per-TP benefit of `clv * retention_rate - campaign_cost = £14,500` swamps
the £500 per-FP contact cost, so the optimiser's preferred policy is to
contact essentially every customer.

This is mathematically correct but operationally unrealistic. Section 11
prints the resulting operational profile (contact rate, contacts per saved
customer, campaign spend) so the headline figures are not read in isolation.
A production deployment would either (a) tighten the cost assumptions, or
(b) introduce an explicit contact-budget constraint and pick the highest
threshold that still respects it.

## 4. Cross-Validated Robustness Check

The full pipeline (SMOTE plus Random Forest plus threshold optimisation)
is re-run inside 5-fold stratified cross-validation on the training fold.
The mean and standard deviation of the cost-optimal threshold quantify how
stable the choice is across folds.

In [ ]:
cv = cv_threshold(X_train, y_train, n_splits=5, costs=costs, random_state=RANDOM_STATE)
print(f"CV optimal threshold : {cv.threshold_mean:.3f} +/- {cv.threshold_std:.3f}")
print(f"CV recall @ optimal  : {cv.recall_mean:.3f} +/- {cv.recall_std:.3f}")
print(f"CV cost at optimal   : GBP {cv.cost_mean:>13,.0f} +/- GBP {cv.cost_std:,.0f}")

## 5. Sensitivity Analysis

The cost-optimal threshold depends on three assumed cost parameters: the
cost of missing a churner (CLV), the cost of a retention contact, and the
probability that a retention campaign succeeds. Each one is varied
independently below.

In [ ]:
records = []
for clv in [10_000, 25_000, 50_000, 75_000, 100_000]:
    c = CostMatrix(clv=clv, campaign_cost=500, retention_rate=0.3)
    t, total = optimal_threshold(y_val.values, y_val_prob, c)
    records.append({"CLV (GBP)": clv, "t*": t, "Val cost (GBP)": total})
pd.DataFrame(records)

In [ ]:
records = []
for cc in [100, 250, 500, 1_000, 2_500]:
    c = CostMatrix(clv=50_000, campaign_cost=cc, retention_rate=0.3)
    t, total = optimal_threshold(y_val.values, y_val_prob, c)
    records.append({"Campaign GBP": cc, "t*": t, "Val cost (GBP)": total})
pd.DataFrame(records)

In [ ]:
for ret in [0.10, 0.20, 0.30, 0.40, 0.50]:
    ev = expected_value(y_val.values, y_val_prob, threshold=t_star, retention_rate=ret)
    print(f"retention rate = {ret:.0%}    EV on val fold = GBP {ev:>14,.0f}")

The cost-optimal threshold rises with CLV (fewer false positives are
preferred when missing a churner is cheaper) and rises with campaign cost
(contacts get more expensive). The expected value is positive only when
the retention campaign actually retains around 20% or more of contacted
customers.

## 6. Model Comparison

Three classifiers, all wrapped in an `imblearn.Pipeline` with SMOTE on the
training fold, evaluated at `t*` selected on the validation fold and
scored on the test fold.

In [ ]:
scaler = StandardScaler().fit(X_train)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=5,
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
}

rows = []
for name, clf in models.items():
    steps = [('smote', SMOTE(random_state=RANDOM_STATE))]
    if name == 'Logistic Regression':
        steps.append(('scale', StandardScaler()))
    steps.append(('clf', clf))
    pipeline = ImbPipeline(steps=steps)
    pipeline.fit(X_train, y_train)
    p_test = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (p_test >= t_star).astype(int)
    rows.append({
        'Model': name,
        'AUC-ROC (test)': roc_auc_score(y_test, p_test),
        'AUC-PR (test)':  average_precision_score(y_test, p_test),
        f'Recall @ {t_star:.2f}':    recall_score(y_test, y_pred),
        f'Precision @ {t_star:.2f}': precision_score(y_test, y_pred),
    })

results_df = pd.DataFrame(rows)
print(results_df.round(3).to_string(index=False))

AUC-ROC differences of order 0.01 are within bootstrap noise on
n ~ 2,900. Tree-based models outperform Logistic Regression on AUC-PR
(the more informative metric on a 10% positive dataset), indicating
non-linear interactions among predictors.

## 7. Permutation Feature Importance

Computed on the test fold using the frozen Random Forest pipeline.

In [ ]:
perm = permutation_importance(
    pipe, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
imp = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(imp)), imp['importance'][::-1], xerr=imp['std'][::-1], color='steelblue')
ax.set_yticks(range(len(imp)))
ax.set_yticklabels(imp['feature'][::-1])
ax.set_xlabel('Mean decrease in score')
ax.set_title('Permutation importance (test fold, top 15)')
plt.tight_layout(); plt.show()
imp

Anonymised campaign-acquisition codes (`origin_up_*`) and channel codes
(`channel_*`) dominate the importance ranking. Absolute price features
appear but rank below.

This is a classifier-side finding. The Cox model in the next section
uses a different covariate set and supports a different finding.

## 8. Survival Analysis

Churn is reframed as a time-to-event problem. Heavy-tailed covariates are
log-transformed so the hazard ratios are interpretable as percentage
changes. The Schoenfeld residual test checks the proportional-hazards
assumption.

In [ ]:
from src.survival import prepare_cox_frame, fit_cox

covariates = ['cons_12m', 'net_margin', 'var_year_price_off_peak', 'has_gas']
cox_df = prepare_cox_frame(df, covariates=covariates, duration_col='tenure', event_col='churn')
report = fit_cox(cox_df, duration_col='tenure', event_col='churn', penalizer=0.01)

print(f"Concordance index: {report.concordance:.3f}")
print(f"Proportional-hazards assumption holds (all p > 0.05): {report.ph_assumption_holds}\n")
print('Cox summary:')
print(report.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].round(3))
print('\nSchoenfeld residual test:')
print(report.schoenfeld[['test_statistic', 'p']].round(3))

A concordance index of around 0.56 means the Cox model only marginally
separates eventual churners from non-churners on tenure alone. `has_gas`
shows HR around 0.90 (p around 0.04): dual-fuel customers have about 10%
lower churn hazard, statistically significant but moderate in magnitude.

If the Schoenfeld test rejects PH for any covariate, the constant
hazard-ratio interpretation is unreliable for that covariate; a stratified
Cox or a time-varying coefficient model is the appropriate next step.

## 9. Final Test-Fold Evaluation

Reported once, on the test fold, at the threshold selected on the
validation fold.

In [ ]:
baseline = RandomForestClassifier(
    n_estimators=100, max_depth=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
baseline.fit(X_train, y_train)
p_base = baseline.predict_proba(X_test)[:, 1]
y_pred_base = (p_base >= 0.5).astype(int)

y_pred_final = (y_test_prob >= t_star).astype(int)

baseline_cost = expected_cost(y_test.values, p_base, 0.5, costs)
final_cost    = expected_cost(y_test.values, y_test_prob, t_star, costs)

rows = [
    {'Stage': 'Baseline (RF, t=0.5)',
     'Recall': recall_score(y_test, y_pred_base),
     'Precision': precision_score(y_test, y_pred_base),
     'Test-fold cost (GBP)': baseline_cost},
    {'Stage': f'Final (SMOTE+RF, t={t_star:.2f})',
     'Recall': recall_score(y_test, y_pred_final),
     'Precision': precision_score(y_test, y_pred_final),
     'Test-fold cost (GBP)': final_cost},
]
pd.DataFrame(rows)

In [ ]:
smote_only = make_smote_rf(random_state=RANDOM_STATE).fit(X_train, y_train)
p_smote_only = smote_only.predict_proba(X_test)[:, 1]
smote_only_cost = expected_cost(y_test.values, p_smote_only, 0.5, costs)

gain_smote = baseline_cost - smote_only_cost
gain_threshold = smote_only_cost - final_cost
ratio = gain_threshold / gain_smote if gain_smote else float('inf')

print(f"Baseline cost      : GBP {baseline_cost:>13,.0f}")
print(f"+ SMOTE only       : GBP {smote_only_cost:>13,.0f}    (delta GBP {gain_smote:>+13,.0f})")
print(f"+ Threshold opt    : GBP {final_cost:>13,.0f}    (delta GBP {gain_threshold:>+13,.0f})")
print(f"\nThreshold optimisation contributed {ratio:.1f}x the cost reduction of SMOTE alone (test fold).")

## 10. Probability Calibration

SMOTE rebalances the training class, but the resulting Random Forest scores
are no longer calibrated to the true (imbalanced) base rate. Wrapping the
SMOTE + RF pipeline in `CalibratedClassifierCV` brings the predicted
probabilities back to the natural prevalence, so the cost-optimal threshold
lives on a meaningful probability scale rather than reflecting an artefact
of the resampling.

In [ ]:
from src.calibration import fit_calibrated_smote_rf, calibration_curve_points

calibrated_pipe = fit_calibrated_smote_rf(
    X_train, y_train, method='isotonic', cv=5, random_state=RANDOM_STATE,
)
y_val_prob_cal = calibrated_pipe.predict_proba(X_val)[:, 1]
y_test_prob_cal = calibrated_pipe.predict_proba(X_test)[:, 1]

print(f'Mean predicted churn prob (validation, uncalibrated): {y_val_prob.mean():.3f}')
print(f'Mean predicted churn prob (validation, calibrated)  : {y_val_prob_cal.mean():.3f}')
print(f'Empirical churn rate (validation)                   : {y_val.mean():.3f}')

In [ ]:
# Reliability diagram: per-bin mean predicted vs. mean observed.
mp_unc, mo_unc, _ = calibration_curve_points(y_val.values, y_val_prob, n_bins=10)
mp_cal, mo_cal, _ = calibration_curve_points(y_val.values, y_val_prob_cal, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Perfect calibration')
ax.plot(mp_unc, mo_unc, marker='o', label='Uncalibrated SMOTE+RF')
ax.plot(mp_cal, mo_cal, marker='o', label='Calibrated (isotonic)')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Empirical churn rate')
ax.set_title('Reliability diagram (validation fold)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Re-pick the cost-optimal threshold on calibrated validation scores.
t_star_cal, val_cost_cal = optimal_threshold(y_val.values, y_val_prob_cal, costs)
print(f'Uncalibrated optimal threshold: {t_star:.3f}    (val cost: GBP {val_cost_star:,.0f})')
print(f'Calibrated   optimal threshold: {t_star_cal:.3f}    (val cost: GBP {val_cost_cal:,.0f})')

The calibrated threshold is typically higher than the uncalibrated one
because isotonic regression shrinks the over-confident SMOTE-distorted scores
back toward the natural prevalence. The two thresholds correspond to roughly
the same decision policy in expected-cost terms; the calibrated version is
just easier to communicate to stakeholders because it lives on a real
probability scale.

## 11. Bootstrap Confidence Intervals on Headline Numbers

The cross-validated threshold from section 4 already carries a mean and a
standard deviation. Recall, precision and the GBP cost on the test fold are
still reported as single point estimates. The block below attaches
percentile-based 95% confidence intervals to all three so the headline
figures come with honest uncertainty bounds.

In [ ]:
from src.uncertainty import bootstrap_metric

# Use calibrated test-fold scores at the calibrated optimal threshold.
recall_ci = bootstrap_metric(
    y_test.values, y_test_prob_cal, threshold=t_star_cal,
    metric='recall', n_bootstrap=2_000, random_state=RANDOM_STATE,
)
precision_ci = bootstrap_metric(
    y_test.values, y_test_prob_cal, threshold=t_star_cal,
    metric='precision', n_bootstrap=2_000, random_state=RANDOM_STATE,
)
cost_ci = bootstrap_metric(
    y_test.values, y_test_prob_cal, threshold=t_star_cal,
    metric='expected_cost', costs=costs,
    n_bootstrap=2_000, random_state=RANDOM_STATE,
)

print('Test-fold headline numbers (calibrated SMOTE+RF, threshold = {:.3f}):'.format(t_star_cal))
print(f'  Recall              : {recall_ci.point_estimate:.3f}    95% CI [{recall_ci.ci_lo:.3f}, {recall_ci.ci_hi:.3f}]')
print(f'  Precision           : {precision_ci.point_estimate:.3f}    95% CI [{precision_ci.ci_lo:.3f}, {precision_ci.ci_hi:.3f}]')
print(f'  Expected cost (GBP) : {cost_ci.point_estimate:>12,.0f}    95% CI [{cost_ci.ci_lo:>12,.0f}, {cost_ci.ci_hi:>12,.0f}]')

In [ ]:
# Visualise the bootstrap distributions.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, res, title in zip(
    axes,
    [recall_ci, precision_ci, cost_ci],
    ['Recall', 'Precision', 'Expected cost (GBP)'],
):
    ax.hist(res.samples, bins=40, alpha=0.7)
    ax.axvline(res.point_estimate, color='C3', linestyle='--', label='point estimate')
    ax.axvline(res.ci_lo, color='C2', linestyle=':', label='95% CI')
    ax.axvline(res.ci_hi, color='C2', linestyle=':')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Operational profile at the chosen threshold

The bootstrap CIs above answer 'how confident are we in the recall / cost /
precision numbers?'. The cell below answers 'what does the policy actually
look like in production?' -- contact rate, contacts per saved customer,
campaign spend. The point is to keep the recall-1.0 result honest: at these
cost assumptions, recall is 1.0 because the policy is effectively 'contact
everyone'.

In [ ]:
from src.evaluation import confusion_counts

# Use the calibrated test-fold probabilities at the calibrated optimal threshold.
y_pred_op = (y_test_prob_cal >= t_star_cal).astype(int)
tn, fp, fn, tp = confusion_counts(y_test.values, y_pred_op)

n = len(y_test)
contacted = tp + fp
contact_rate = contacted / n
expected_saves = tp * costs.retention_rate
contacts_per_save = contacted / expected_saves if expected_saves else float('inf')
spend = contacted * costs.campaign_cost
expected_savings = -expected_cost(y_test.values, y_test_prob_cal, t_star_cal, costs)

print(f'Operational profile (calibrated SMOTE+RF, threshold = {t_star_cal:.3f}):')
print(f'  test-fold size     : {n:,}')
print(f'  contacted          : {contacted:,}   ({contact_rate:.1%})')
print(f'  true positives     : {tp:,}')
print(f'  false positives    : {fp:,}')
print(f'  false negatives    : {fn:,}')
print(f'  expected saves     : {expected_saves:.1f}   (= TP * retention_rate)')
print(f'  contacts per save  : {contacts_per_save:.1f}')
print(f'  campaign spend     : GBP {spend:>10,.0f}')
print(f'  expected savings   : GBP {expected_savings:>10,.0f}')

Read this together with the bootstrap CIs and the cohort-split result. A
contact rate near 100% and contacts-per-save above 30 says the cost
optimiser is exploiting the very generous per-TP benefit assumption. The
recall-1.0 result is real but it is not free, and the operating point is
brittle to any sensible adjustment of the cost parameters or to any
realistic operational constraint (campaign capacity, contact fatigue, opt-
out rates). Do not deploy this threshold without re-running the sensitivity
analysis under tighter assumptions.

## 12. Cohort-Based Validation

The data is a single 2015 snapshot, so a true out-of-time hold-out is not
possible: every customer's churn label is observed at the same calendar
moment. The strongest substitute the data supports is a cohort split on
`date_activ` -- the model trains on the earlier-activated customers and is
evaluated on the more-recently activated ones.

This tests **cross-cohort** generalisation. It does not test temporal
generalisation of the churn outcome itself; it only checks that the model
trained on one acquisition cohort still works on the next.

In [ ]:
from src.time_split import cohort_split, cohort_summary

# Run the cohort split on the modelling dataset and re-attach the engineered features.
# The model dataset (df) already has the engineered columns from notebook 02.
# Lower tenure means more recent activation, so we send the lowest-tenure 20% to the test fold.
cohort_train_df, cohort_test_df = cohort_split(
    df, sort_col='tenure', test_quantile=0.2, test_is_above=False,
)

cohort_summary(cohort_train_df, cohort_test_df, sort_col='tenure')

In [ ]:
# Fit the same pipeline on the earlier cohort and score the later cohort.
X_co_tr, y_co_tr = split_features_target(cohort_train_df)
X_co_te, y_co_te = split_features_target(cohort_test_df)

cohort_pipe = make_smote_rf(random_state=RANDOM_STATE).fit(X_co_tr, y_co_tr)
y_co_prob = cohort_pipe.predict_proba(X_co_te)[:, 1]

cohort_auc = roc_auc_score(y_co_te, y_co_prob)
cohort_recall_at_tstar = recall_score(y_co_te, (y_co_prob >= t_star).astype(int))
print(f'Cohort-test AUC-ROC     : {cohort_auc:.3f}')
print(f'Cohort-test recall at t*: {cohort_recall_at_tstar:.3f}')

A meaningful drop in AUC or recall here would indicate that the customer
acquisition pattern changed enough to shift the churn drivers. A roughly
equal number to the random-split test fold suggests the model is not
over-fit to any particular acquisition cohort. Either way, this is a weaker
check than true out-of-time validation, and the result must be read with
that caveat in mind.

## 13. Uplift / Heterogeneous-Treatment-Effect Framing

The classifier above answers the question *who is likely to churn?*. The
question the retention team actually needs answered is *who will respond to
a retention contact?*. The two are different: a customer can be a high-risk
churner who would leave regardless of any offer, or a low-risk customer
whose loyalty is in fact contingent on continued attention. Targeting purely
on predicted churn probability spends campaign budget on the first group
and ignores the second.

The right modelling object is the **conditional average treatment effect**:

> CATE(x) = E[Y(retain=1) - Y(retain=0) | X = x]

estimated by an X-learner, R-learner or doubly-robust learner. Libraries
such as `econml` and `causalml` provide ready implementations.

**Why this notebook does not fit one.** The PowerCo dataset is purely
observational: it has churn outcomes and customer features, but no
treatment indicator -- no record of which customers were contacted with a
retention offer or what happened to them as a result. Fitting an uplift
model on this snapshot would require simulating a treatment column, which
would produce a result without any external validity.

**What it would take to make uplift modelling defensible here.** Run a
randomised retention pilot on, say, 5,000 customers stratified by predicted
churn risk -- contact half with the offer, leave the other half alone, and
record the realised retention outcome for both groups. With that data in
hand, an X-learner or R-learner fit on the joined features-plus-treatment-
plus-outcome table would estimate CATE and the operating threshold could
shift from "high churn probability" to "high expected uplift per pound
spent".

This is the recommended next step for any production deployment of this
model. Continuing to tune the existing classifier on this snapshot yields
diminishing returns; the uplift framing addresses a different and more
business-relevant question.

## 14. Limitations

1. The data is a single 2015 snapshot. The test-fold cost reduction is a
   one-shot estimate, not an annualised figure, and there is no
   out-of-time validation.
2. The cost parameters (CLV £50k, campaign £500, retention benefit £10k)
   are assumed values. The sensitivity sweep above is the right way to
   express how outputs respond to those assumptions.
3. SMOTE rebalances the training class only; the test fold remains at the
   natural prevalence. Probability calibration after SMOTE is a known
   issue and would need isotonic or Platt scaling before deployment.
4. The dataset shows no evidence that absolute price level is a primary
   driver of churn in this snapshot. Price changes, competitor pricing
   and contract-change history are not present in the data, so this is
   not a refutation of price sensitivity in general.
5. A threshold of around 0.05 contacts a large fraction of the customer
   base. This is an upper bound on recall under the assumed cost matrix;
   any production deployment would need contact-fatigue constraints that
   are not modelled here.